# CH101 Production Blockout v010 automation

이 Notebook은 CH101 승인 2D 시트를 읽기 입력으로 삼아 LOD·2-bone production-review weight와 idle/run/attack 변형 검토 렌더를 포함한 기술용 Blockout을 만들고, 렌더·FBX·JSON 검증 결과를 Google Drive에 백업한다. 최종 모델과 Unity Humanoid 확인은 아직 남아 있다.

In [ ]:
from pathlib import Path
import shutil
import subprocess
from IPython.display import Image, display

REPO_DIR = Path('/content/re-camp')
TOOLS_DIR = Path('/content/re-camp-blender')
DRIVE_ROOT = Path('/content/drive/MyDrive/re-camp')
WORK_ROOT = Path('/content/re-camp-runtime/CH101_v010')
SOURCE_BRANCH = 'current/art-roster-gate-a-ch102'
SOURCE_COMMIT = 'b6c9b3128358e061eee6184230929413eba84101'
SOURCE_REFERENCE = 'art_refs/characters/rin/concept/CH101_Rin_CharacterSheet_APPROVED_v001.png'
WORK_ROOT.mkdir(parents=True, exist_ok=True)
REFERENCE = REPO_DIR / SOURCE_REFERENCE
BUILD_SCRIPT = TOOLS_DIR / 'scripts/blender/build_blockout.py'
VALIDATE_SCRIPT = TOOLS_DIR / 'scripts/blender/validate_asset.py'
if not REFERENCE.is_file():
    raise FileNotFoundError(REFERENCE)
display(Image(filename=str(REFERENCE), width=420))
print('Source:', REFERENCE)
print('Source commit lock:', SOURCE_COMMIT)
print('Blockout revision: v010')

In [ ]:
launcher = ['xvfb-run', '-a'] if shutil.which('xvfb-run') else []
build_command = launcher + [
    'blender', '--background', '--python', str(BUILD_SCRIPT), '--',
    '--character', 'CH101',
    '--source-asset', str(REFERENCE),
    '--source-commit', SOURCE_COMMIT,
    '--output-dir', str(WORK_ROOT),
    '--render', '--export-fbx',
    '--optimize-budget', '--generate-lods', '--production-skinning-review',
]
run = subprocess.run(build_command, capture_output=True, text=True)
print('Launcher:', launcher or ['blender'])
print(run.stdout[-5000:])
if run.returncode != 0:
    print(run.stderr[-5000:])
    raise RuntimeError(f'Blender blockout failed: {run.returncode}')

In [ ]:
for view in ('front', 'side', 'back'):
    image_path = WORK_ROOT / 'renders' / f'{view}.png'
    if image_path.exists():
        print(view)
        display(Image(filename=str(image_path), width=320))
    else:
        print('Missing render:', image_path)

In [ ]:
blend_path = WORK_ROOT / 'CH101_Blockout_REVIEW_v010.blend'
validation_report = WORK_ROOT / 'reports' / 'CH101_Blockout_validation_v010.json'
validate_command = launcher + [
    'blender', '--background', '--python', str(VALIDATE_SCRIPT), '--',
    '--blend', str(blend_path),
    '--report', str(validation_report),
]
run = subprocess.run(validate_command, capture_output=True, text=True)
print(run.stdout[-5000:])
if run.returncode != 0:
    print(run.stderr[-5000:])
    raise RuntimeError(f'Blockout validation failed: {run.returncode}')

In [ ]:
drive_output = DRIVE_ROOT / 'ch101_blockout'
drive_output.mkdir(parents=True, exist_ok=True)
shutil.copytree(WORK_ROOT, drive_output, dirs_exist_ok=True)
print('Backed up to:', drive_output)
print('Saved files:')
for path in sorted(drive_output.rglob('*')):
    if path.is_file():
        print(path.relative_to(drive_output))

## 판정 경계

이 실행이 성공해도 CH101 Gate B는 승인되지 않는다. 실제 3D 비율·관절 충돌·Material·Motion·Unity Import·Android 증거가 생길 때 별도 검토한다.